# Lab 2: Aerial Image Land Use Classification (UC Merced, 21 classes)

Classifies 256 x 256 aerial orthoimagery tiles into 21 land use classes. Compares a small
CNN trained from scratch with transfer learning from an ImageNet pretrained Inception v3.

**Run in Google Colab with a GPU** (Runtime > Change runtime type > GPU).

In [ ]:
# Download the dataset (about 330 MB). If the university link is down, the same dataset is on Kaggle
# ("UC Merced Land Use") and in TensorFlow Datasets as "uc_merced".
!wget -q http://weegee.vision.ucmerced.edu/datasets/UCMerced_LandUse.zip
!unzip -q -o UCMerced_LandUse.zip

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import inception_v3, Inception_V3_Weights

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "./UCMerced_LandUse/Images"
print("Device:", device)

## Train and test split with separate transforms

In [ ]:
def make_loaders(train_tf, test_tf, valid_size=0.2, batch_size=64):
    train_ds = datasets.ImageFolder(DATA_DIR, transform=train_tf)
    test_ds = datasets.ImageFolder(DATA_DIR, transform=test_tf)
    idx = np.random.RandomState(SEED).permutation(len(train_ds))
    split = int(valid_size * len(idx))
    test_idx, train_idx = idx[:split], idx[split:]
    train_loader = DataLoader(Subset(train_ds, train_idx), batch_size=batch_size, shuffle=True, num_workers=2)
    test_loader = DataLoader(Subset(test_ds, test_idx), batch_size=batch_size, shuffle=False, num_workers=2)
    return train_loader, test_loader, train_ds.classes

# Augmented training images; plain test images. Same split for both models.
cnn_train_tf = transforms.Compose([
    transforms.Resize(192),
    transforms.RandomResizedCrop(96, scale=(0.5, 1.0)),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])
cnn_test_tf = transforms.Compose([transforms.Resize((96, 96)), transforms.ToTensor()])

train_loader, test_loader, classes = make_loaders(cnn_train_tf, cnn_test_tf)
print(len(classes), "classes:", classes[:5], "...")

## Shared training and evaluation helpers

In [ ]:
def train_model(model, loader, optimizer, epochs, aux_weight=None):
    criterion = nn.CrossEntropyLoss()
    losses = []
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            out = model(inputs)
            if aux_weight is not None and isinstance(out, tuple):   # Inception returns (logits, aux_logits) in train mode
                loss = criterion(out[0], labels) + aux_weight * criterion(out[1], labels)
            else:
                loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            running += loss.item()
        losses.append(running / len(loader))
        print(f"Epoch {epoch + 1}: loss {losses[-1]:.4f}")
    return losses


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        preds = model(inputs).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return correct / total

## Model 1: Simple CNN from scratch

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=21):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 24 * 24, 512)
        self.fc2 = nn.Linear(512, num_classes)
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 96 -> 48
        x = self.pool(F.relu(self.conv2(x)))   # 48 -> 24
        x = torch.flatten(x, 1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)

cnn = SimpleCNN(len(classes)).to(device)
cnn_losses = train_model(cnn, train_loader, optim.Adam(cnn.parameters(), lr=1e-3), epochs=10)
cnn_acc = evaluate(cnn, test_loader)
print(f"Simple CNN test accuracy: {cnn_acc:.1%}")

## Model 2: Transfer learning with Inception v3

Inception v3 expects 299 x 299 inputs normalised with ImageNet statistics. The pretrained
backbone is frozen and only the new classification heads are trained.

In [ ]:
weights = Inception_V3_Weights.DEFAULT
imagenet_norm = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
inc_train_tf = transforms.Compose([
    transforms.Resize(320), transforms.RandomResizedCrop(299, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(), imagenet_norm,
])
inc_test_tf = transforms.Compose([transforms.Resize((299, 299)), transforms.ToTensor(), imagenet_norm])
inc_train_loader, inc_test_loader, _ = make_loaders(inc_train_tf, inc_test_tf, batch_size=32)

inception = inception_v3(weights=weights, aux_logits=True)
for p in inception.parameters():
    p.requires_grad = False
inception.fc = nn.Linear(inception.fc.in_features, len(classes))
inception.AuxLogits.fc = nn.Linear(inception.AuxLogits.fc.in_features, len(classes))
inception = inception.to(device)

head_params = list(inception.fc.parameters()) + list(inception.AuxLogits.fc.parameters())
inc_losses = train_model(inception, inc_train_loader, optim.Adam(head_params, lr=1e-3), epochs=5, aux_weight=0.4)
inc_acc = evaluate(inception, inc_test_loader)
print(f"Inception v3 transfer learning test accuracy: {inc_acc:.1%}")

## Compare

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(cnn_losses) + 1), cnn_losses, label=f"Simple CNN (test acc {cnn_acc:.1%})")
plt.plot(range(1, len(inc_losses) + 1), inc_losses, label=f"Inception v3 transfer (test acc {inc_acc:.1%})")
plt.xlabel("Epoch"); plt.ylabel("Training loss"); plt.legend()
plt.title("UC Merced land use classification")
plt.tight_layout(); plt.show()